In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 4
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.352686241793089

Trial 1 =========================================
13.141741373215794

Trial 2 =========================================
16.195851566102064

Trial 3 =========================================
13.458869597525757

Trial 4 =========================================
16.232929174556286

Trial 5 =========================================
14.294517397015426

Trial 6 =========================================
18.784584694499554

Trial 7 =========================================
15.929222010399386

Trial 8 =========================================
13.42776807226306

Trial 9 =========================================
14.263985945667283

Trial 10 =========================================
17.426634076559182

Trial 11 =========================================
14.413592687223856

Trial 12 =========================================
16.19641188801746

Trial 13 =========================================
15.973061503078208

Trial 14 =========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 17 =========================================
14.268361428699828

Trial 18 =========================================
13.433791003665176

Trial 19 =========================================
14.447275688387592



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 20 =========================================
16.10317702746093

Trial 21 =========================================
16.20095428116147

Trial 22 =========================================
13.872370116998514

Trial 23 =========================================
15.37391829364507

Trial 24 =========================================
14.343588036794204

Trial 25 =========================================
15.367709796899183

Trial 26 =========================================
13.912069880018791

Trial 27 =========================================
17.090078782701607

Trial 28 =========================================
14.397354141582221

Trial 29 =========================================
17.50150489992133

Trial 30 =========================================
14.442838658618363

Trial 31 =========================================
14.452455044323163

Trial 32 =========================================
14.373013330470956

Trial 33 =========================================
13.459108047795123

Trial 34 =

...

Trial 38 =========================================
16.233984449523852

Trial 39 =========================================
14.278166377702979

Trial 40 =========================================
17.51080791048812

Trial 41 =========================================
15.870449521271267

Trial 42 =========================================
13.447334238996755

Trial 43 =========================================
15.645480521841339

Trial 44 =========================================
18.678587606756548

Trial 45 =========================================
16.24341108329967

Trial 46 =========================================
15.872968707021332

Trial 47 =========================================
14.091915692699486

Trial 48 =========================================
15.388148247095431

Trial 49 =========================================
16.25794350146211

Trial 50 =========================================
14.162059941178049

Trial 51 =========================================
13.876816516505848

Trial 52 

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 57 =========================================
16.26909164223956

Trial 58 =========================================
15.400562576408475

Trial 59 =========================================
15.352859992968924

Trial 60 =========================================
15.399283535509051



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/mi

Trial 61 =========================================
15.977539836672863

Trial 62 =========================================
14.638197298435706

Trial 63 =========================================
14.629915358827638

Trial 64 =========================================
15.39234671514194

Trial 65 =========================================
13.441219307940685

Trial 66 =========================================
15.361321275872335

Trial 67 =========================================
13.44745740630655

Trial 68 =========================================
15.16160130856245

Trial 69 =========================================
14.816364908094112

Trial 70 =========================================
15.38976968665937

Trial 71 =========================================
14.636875343489674

Trial 72 =========================================
16.23091661510112

Trial 73 =========================================
14.84762916326765

Trial 74 =========================================
13.868019177845298

Trial 75 ===

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.802439124714937
Avg = 15.358819650025412
Std = 1.3287949173438443


In [6]:
print(y_max_arr.tolist())

[15.352686241793089, 13.141741373215794, 16.195851566102064, 13.458869597525757, 16.232929174556286, 14.294517397015426, 18.784584694499554, 15.929222010399386, 13.42776807226306, 14.263985945667283, 17.426634076559182, 14.413592687223856, 16.19641188801746, 15.973061503078208, 14.065594393173106, 15.319121869807544, 14.855671621468327, 14.268361428699828, 13.433791003665176, 14.447275688387592, 16.10317702746093, 16.20095428116147, 13.872370116998514, 15.37391829364507, 14.343588036794204, 15.367709796899183, 13.912069880018791, 17.090078782701607, 14.397354141582221, 17.50150489992133, 14.442838658618363, 14.452455044323163, 14.373013330470956, 13.459108047795123, 16.264972552874937, 16.186251442925716, 14.085872846553029, 16.25072455424016, 16.233984449523852, 14.278166377702979, 17.51080791048812, 15.870449521271267, 13.447334238996755, 15.645480521841339, 18.678587606756548, 16.24341108329967, 15.872968707021332, 14.091915692699486, 15.388148247095431, 16.25794350146211, 14.162059

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-4.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)